In [1]:
"""
=========================================================
PUNTO 4 — COMPARACIÓN DE MODELOS (tablas de rejilla doble)
=========================================================
Parte del fichero 'dataset_60_topado.csv' (subconjunto de 60 pacientes).

Para cada modelo se prueba una rejilla de dos parámetros y se mide
el balanced accuracy POR PACIENTE (folds fijos + agregación por voto),
igual que en el script del SVM.

Modelos y parámetros que se exploran:
  - KNN:                 nº vecinos (k)  x  tipo de peso
  - Random Forest:       nº árboles      x  profundidad máxima
  - XGBoost:             profundidad     x  learning rate
  - Regresión logística: C               x  (l2)

REQUISITO: tener instalado xgboost.
  En el notebook, ejecuta en una celda:   !pip install xgboost
  (o en Anaconda Prompt:  conda install -c conda-forge xgboost)

CÓMO USARLO:
  Cambia 'path_dataset' a la ruta de tu ordenador y ejecuta.
=========================================================
"""

import pandas as pd
import numpy as np
from collections import Counter

from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import balanced_accuracy_score
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTETomek

import warnings
warnings.filterwarnings('ignore')

# =========================================================
# RUTA — cámbiala a la de tu ordenador
# =========================================================
path_dataset = r"C:\Users\josem\Desktop\tfg\dataset_60_topado.csv"

N_FEATURES = 30   # nº de features a usar (las más importantes según Random Forest)

# =========================================================
# 1. CARGAR DATASET
# =========================================================
d = pd.read_csv(path_dataset, sep=';', decimal=',')
meta = ['archivo', 'diagnostico', 'ventana', 'pid', 'fold']
feats = [c for c in d.columns if c not in meta]
for c in feats:
    d[c] = pd.to_numeric(d[c], errors='coerce')
d = d.dropna(subset=feats)

# =========================================================
# 2. SELECCIONAR LAS 30 FEATURES MÁS IMPORTANTES
# =========================================================
rf_imp = RandomForestClassifier(n_estimators=500, class_weight='balanced_subsample',
                                random_state=42, n_jobs=-1)
rf_imp.fit(MinMaxScaler().fit_transform(d[feats]), d['diagnostico'].values)
importancias = pd.Series(rf_imp.feature_importances_, index=feats).sort_values(ascending=False)
top_feats = list(importancias.index[:N_FEATURES])

# =========================================================
# 3. FUNCIÓN DE VALIDACIÓN POR PACIENTE
#    (folds fijos + escalado + SMOTE en train + voto)
# =========================================================
def cv_por_paciente(crear_modelo, usar_label_encoder=False):
    preds = []
    le = LabelEncoder().fit(d['diagnostico']) if usar_label_encoder else None

    for fv in sorted(d['fold'].unique()):
        tr = d[d['fold'] != fv]
        va = d[d['fold'] == fv].copy()

        X_tr = tr[top_feats].values
        y_tr = tr['diagnostico'].values
        X_va = va[top_feats].values

        sc = MinMaxScaler()
        X_tr = sc.fit_transform(X_tr)
        X_va = sc.transform(X_va)

        k = min(5, min(Counter(y_tr).values()) - 1)
        if k >= 1:
            smt = SMOTETomek(smote=SMOTE(k_neighbors=k, random_state=42), random_state=42)
            X_tr, y_tr = smt.fit_resample(X_tr, y_tr)

        modelo = crear_modelo()
        if usar_label_encoder:   # XGBoost necesita etiquetas numéricas
            modelo.fit(X_tr, le.transform(y_tr))
            pred = le.inverse_transform(modelo.predict(X_va))
        else:
            modelo.fit(X_tr, y_tr)
            pred = modelo.predict(X_va)

        va['pred'] = pred
        for paciente, grupo in va.groupby('pid'):
            diag_pred = Counter(grupo['pred']).most_common(1)[0][0]
            preds.append({'real': grupo['diagnostico'].iloc[0], 'pred': diag_pred})

    r = pd.DataFrame(preds)
    return balanced_accuracy_score(r['real'], r['pred'])

# =========================================================
# 4. FUNCIÓN PARA IMPRIMIR UNA TABLA DE REJILLA DOBLE
# =========================================================
def tabla(nombre, filas, cols, etiq_fila, etiq_col, crear, usar_le=False):
    print(f"\n{'='*55}")
    print(f"  {nombre}  (balanced accuracy por paciente)")
    print(f"{'='*55}")
    print(f"{etiq_fila + ' \\ ' + etiq_col:<16}", end='')
    for c in cols:
        print(f"{str(c):>10}", end='')
    print()
    for f in filas:
        print(f"{str(f):<16}", end='')
        for c in cols:
            bal = cv_por_paciente(lambda: crear(f, c), usar_le)
            print(f"{bal:>10.3f}", end='')
        print()

# =========================================================
# 5. LAS CUATRO TABLAS
# =========================================================

# --- KNN: nº de vecinos (k) x tipo de peso ---
tabla("KNN", [3, 5, 7, 9], ['uniform', 'distance'], 'k', 'peso',
      lambda k, w: KNeighborsClassifier(n_neighbors=k, weights=w))

# --- RANDOM FOREST: nº de árboles x profundidad máxima ---
tabla("RANDOM FOREST", [100, 300, 500], [5, 10, None], 'n_arb', 'prof',
      lambda n, p: RandomForestClassifier(n_estimators=n, max_depth=p,
                                          class_weight='balanced_subsample',
                                          random_state=42, n_jobs=-1))

# --- XGBOOST: profundidad x learning rate ---
tabla("XGBOOST", [3, 5, 7], [0.05, 0.1, 0.3], 'prof', 'lr',
      lambda p, l: XGBClassifier(max_depth=p, learning_rate=l, n_estimators=200,
                                 random_state=42, verbosity=0),
      usar_le=True)

# --- REGRESIÓN LOGÍSTICA: C x penalización ---
tabla("LOGISTIC REGRESSION", [0.1, 1, 10], ['l2'], 'C', 'pen',
      lambda c, pen: LogisticRegression(C=c, penalty=pen, max_iter=1000,
                                        class_weight='balanced', random_state=42))


  KNN  (balanced accuracy por paciente)
k \ peso           uniform  distance
3                    0.567     0.567
5                    0.600     0.617
7                    0.617     0.617
9                    0.633     0.617

  RANDOM FOREST  (balanced accuracy por paciente)
n_arb \ prof             5        10      None
100                  0.633     0.617     0.600
300                  0.617     0.633     0.617
500                  0.617     0.650     0.633

  XGBOOST  (balanced accuracy por paciente)
prof \ lr             0.05       0.1       0.3
3                    0.567     0.567     0.583
5                    0.550     0.583     0.550
7                    0.567     0.567     0.600

  LOGISTIC REGRESSION  (balanced accuracy por paciente)
C \ pen                 l2
0.1                  0.633
1                    0.617
10                   0.533

FIN — recuerda que el SVM (tabla aparte) daba 0.65 como mejor valor
